# 01 Preprocessing — Stage 1 Dataset Build

This notebook runs the complete Stage 1 preprocessing pipeline and exports processed files.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Standardized project root discovery
root = Path.cwd().resolve()
while root != root.parent and not (root / 'README.md').exists():
    root = root.parent
PROJECT_ROOT = root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.load_data import load_stage1_data
from src.preprocessing.clean import handle_missing
from src.preprocessing.target_builder import build_targets
from src.preprocessing.engineer_features import engineer_features
from src.preprocessing.encode import encode_features

print('Project root:', PROJECT_ROOT)

Project root: C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48


In [2]:
df = load_stage1_data()
print('Loaded shape:', df.shape)
df.head(2)

Loaded CSV file: NFHS5_Individual.csv -> (227391, 32)
Analytic sample: full file (227,391 rows)
Loaded shape: (227391, 32)


C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\src\preprocessing\load_data.py:120: UserWarning: Columns not found in CSV and skipped: ['v467f', 'v467i', 'm14']
  warnings.warn(f"Columns not found in CSV and skipped: {missing}")


,caseid,v001,v002,v021,v024,v025,v012,v013,v106,v130,...,v467b,v467c,v467d,v467e,v467g,v467h,v626a,s245a,s245b,s245h
0,0100101305 04,113,5,113,jammu & kashmir,rural,22,20-24,higher,muslim,...,big problem,not a big problem,not a big problem,big problem,big problem,big problem,never had sex,NaN,NaN,NaN
1,0100101305 05,113,5,113,jammu & kashmir,rural,19,15-19,secondary,muslim,...,big problem,big problem,big problem,big problem,big problem,big problem,never had sex,NaN,NaN,NaN


In [3]:
df = handle_missing(df)
df = build_targets(df)
df = engineer_features(df)
encoded = encode_features(df)

print('Post-encoding shape:', encoded.shape)
print(df[['target_household', 'target_logistic', 'target_facility']].mean())

Dropped v466 (100% missing in this extract)


v131: filled 495 missing values with 'missing'
v717: filled 193365 missing values with 'missing'
v169a: filled 193343 missing values with 'missing'
v170: filled 193343 missing values with 'missing'
v159: filled 1 missing values with 'missing'
v743f: filled 204279 missing values with 'missing'
Feature missing values remaining: 0
target_household: positive rate = 0.2183 (21.8%) | built from ['v467b', 'v467c'] | counts = {0: 177748, 1: 49643}
target_logistic: positive rate = 0.2747 (27.5%) | built from ['v467d', 'v467e'] | counts = {0: 164935, 1: 62456}


C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\src\preprocessing\target_builder.py:28: UserWarning: target_household: missing barrier columns ['v467f']; building from available columns only.
  warnings.warn(
C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\src\preprocessing\target_builder.py:28: UserWarning: target_facility: missing barrier columns ['v467i']; building from available columns only.
  warnings.warn(


target_facility: positive rate = 0.4089 (40.9%) | built from ['v467g', 'v467h'] | counts = {0: 134410, 1: 92981}


Post-encoding shape: (227391, 37)
target_household    0.218316
target_logistic     0.274663
target_facility     0.408904
dtype: float64


In [4]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

X_full = encoded.copy()
X_full.to_csv(processed_dir / 'X_features.csv', index=False)
df['target_household'].to_csv(processed_dir / 'y_household.csv', index=False)
df['target_logistic'].to_csv(processed_dir / 'y_logistic.csv', index=False)
df['target_facility'].to_csv(processed_dir / 'y_facility.csv', index=False)

print('Saved processed files to:', processed_dir)
print('X_features shape:', X_full.shape)
for t in ['target_household', 'target_logistic', 'target_facility']:
    print(t, df[t].value_counts().to_dict())

Saved processed files to: C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48\data\processed
X_features shape: (227391, 37)
target_household {0: 177748, 1: 49643}
target_logistic {0: 164935, 1: 62456}
target_facility {0: 134410, 1: 92981}
